# Informationsextraktion und Argument Mining

Bisher haben wir Texte gezählt, verglichen, gruppiert und klassifiziert. Das Ergebnis war immer eine Aussage *über* Dokumente: Dieses Wort ist typisch für die SZ, diese Schlagzeile gehört ins Ressort *Sport*. **Informationsextraktion** (IE) geht einen Schritt weiter. Sie zieht aus unstrukturiertem Text **strukturierte Daten** heraus - Tabellen mit Akteuren, Beziehungen, Ereignissen, Orten und Zeitpunkten -, die sich danach wie jede andere Datenbank auswerten lassen.

Han, Pei und Tong (2023, Kap. 12.1) nennen als typische Teilaufgaben:

- **Named Entity Recognition** (NER): Personen, Organisationen, Orte erkennen - das haben wir in Skript 02 gemacht;
- **Koreferenzauflösung**: erkennen, dass *Merkel*, *die Kanzlerin* und *sie* dieselbe Person meinen;
- **Relationsextraktion**: Beziehungen zwischen Entities finden (*Schäuble ist Finanzminister*, *die CSU kritisiert Merkel*);
- **Ereignisextraktion**: Ereignisse mit ihren Beteiligten, Orten und Zeitpunkten erfassen (*ein Anschlag in Paris im November 2015*);
- **Taxonomien** aufbauen: Begriffe hierarchisch ordnen (*Griechenland* ist ein *Land*, *Google* ist ein *Unternehmen*).

Eine verwandte, jüngere Aufgabe ist das **Argument Mining**: Statt *wer tut was* fragt es *wer begründet was womit* - es sucht Behauptungen, die Gründe dafür und die Texte, in denen überhaupt argumentiert wird.

Alle Verfahren in diesem Skript sind **regelbasiert** oder einfache überwachte Modelle. Sie bauen auf dem Dependenzparsing und den Named Entities aus Skript 02 auf. Das ist Absicht: Regeln sind durchschaubar, und ihre Fehler lassen sich lesen.

Am Ende dieses Skripts kannst du:

- mit dem Dependenzbaum **Rollen** von Personen (*Finanzminister Schäuble*) und **Relationen** zwischen Entities extrahieren;
- **Ereignisse** mit Ort und Zeit aus Schlagzeilen gewinnen und zu einer Chronik verdichten;
- mit **Hearst-Mustern** (*Länder wie Griechenland und Italien*) eine kleine **Taxonomie** aus den Artikeltexten aufbauen;
- argumentierende Leserkommentare erkennen, dabei einen **Störfaktor** (die Länge) aufdecken und in Kommentaren **Behauptung und Begründung** trennen.

## Präambel

Wir laden alle Pakete und Module, die wir im Folgenden benötigen.

In [ ]:
# Standardbibliothek
import re
from collections import Counter, defaultdict

# Datentransformation und Visualisierung
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Paket für linguistische Annotation
import spacy

# Pakete für Merkmalsextraktion, Klassifikation und Evaluation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

# Hilfsfunktionen des Kurses
import textmining as tm

# Darstellungsoptionen für Tabellen und Grafiken
pd.set_option("display.max_colwidth", 110)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## Daten

Für Rollen, Relationen und Ereignisse verwenden wir Schlagzeile und Teaser (`corpus["text"]`), annotiert mit dem mittleren spaCy-Modell. Skript 04 hat diese Texte bereits geparst; `tm.parse_corpus` liest das Ergebnis aus dem Cache. Wer Skript 04 nicht ausgeführt hat, wartet hier einige Minuten.

In [ ]:
# Lade Korpus und Modell und hole die annotierten Texte (aus dem Cache, falls vorhanden)
corpus = tm.load_corpus()
nlp = spacy.load("de_core_news_md")
docs = tm.parse_corpus(nlp, corpus["text"].tolist(), name="texts")
print(f"{len(docs):,} annotierte Texte")

## Rollen: Wer ist wer?

Nachrichten stellen Personen fast immer mit ihrer Funktion vor: *Finanzminister Schäuble*, *US-Präsident Obama*, *Bundestrainer Löw*. Für eine Datenbank ist das eine Relation **Person - hat Rolle - Funktion**, und sie steht ganz explizit im Text.

In Skript 02 haben wir gesehen, dass die deutschen spaCy-Modelle die Labels der TIGER-Baumbank verwenden. In einer Konstruktion wie *Finanzminister Schäuble* ist dort die **Funktion der Kopf** und der Name hängt als Element des Nominalkerns (`nk`) daran. Wir sehen uns das an einem Beispiel an:

In [ ]:
# Ein Beispiel: Zu welchem Wort gehört der Name?
example = nlp("Finanzminister Schelling will die Steuerreform verteidigen")
pd.DataFrame([(t.text, t.pos_, t.dep_, t.head.text, t.ent_type_) for t in example],
             columns=["Token", "Wortart", "Dependenz", "Kopf", "Entity"])

*Schelling* hängt an *Finanzminister*. (Dass das Modell *Finanzminister* selbst als Ort markiert, ist ein Fehler der Entity-Erkennung, der für unsere Regel keine Rolle spielt.) Daraus wird eine einfache Regel: Eine Personen-Entity, deren Kopf das unmittelbar davorstehende Substantiv ist, hat dieses Substantiv als Rolle. Wir sammeln die Rollen für alle Personen im Korpus. Als Name der Person verwenden wir das letzte Wort der Entity - aus *Hans Jörg Schelling* und *Schelling* wird so dieselbe Person. Das ist eine grobe Form der **Koreferenzauflösung**, und sie hat einen Preis: Verschiedene Personen mit demselben Nachnamen fallen zusammen.

In [ ]:
# Sammle Paare aus Person und Rolle: PER-Entity, deren Kopf das Substantiv direkt davor ist
rows = []
for outlet, doc in zip(corpus["outlet"], docs):
    for ent in doc.ents:
        if ent.label_ != "PER" or ent.start == 0:
            continue
        role = doc[ent.start - 1]
        if role.pos_ == "NOUN" and ent.root.head.i == role.i:
            rows.append({"outlet": outlet, "person": ent[-1].text, "role": role.text})
roles = pd.DataFrame(rows)
print(f"{len(roles):,} Rollenangaben für {roles['person'].nunique():,} Personen")

# Zeige die häufigsten Rollen der 15 am häufigsten mit Rolle genannten Personen
top_people = roles["person"].value_counts().head(15).index
(
    roles[roles["person"].isin(top_people)]
    .groupby("person")["role"]
    .agg(lambda r: ", ".join(f"{w} ({n})" for w, n in Counter(r).most_common(4)))
    .reindex(top_people)
    .to_frame("häufigste Rollen")
)

Mit einer Regel aus einer Zeile ist eine kleine Personendatenbank entstanden, und sie ist überwiegend richtig: Merkel ist *Kanzlerin* und *Bundeskanzlerin*, Schäuble *Finanzminister*, Franziskus *Papst*. Drei Dinge zeigen, wo die Arbeit eigentlich beginnt:

- **Varianten derselben Rolle.** *Kanzlerin* und *Bundeskanzlerin*, *Präsident* und *US-Präsident* sind dieselbe Funktion. Eine echte Wissensbasis würde sie auf einen Eintrag abbilden - eine eigene kleine Taxonomie (siehe unten).
- **Eine Person, mehrere Rollen.** Gabriel ist *SPD-Chef*, *Wirtschaftsminister* und *Vizekanzler* zugleich, und andere Personen wechseln im Laufe des Jahres ihr Amt. Eine Relation braucht deshalb oft einen Zeitstempel.
- **Namen sind nicht eindeutig.** Hinter *Müller* oder *Schmidt* stehen viele Personen mit verschiedenen Rollen. Die Aufgabe, eine Nennung der richtigen Person zuzuordnen, heißt **Entity Linking** und braucht Kontext oder eine externe Wissensbasis wie Wikidata.

## Relationen zwischen Entities

In Skript 02 haben wir Tripel aus **Subjekt - Verb - Objekt** extrahiert. Für eine Relationsextraktion im engeren Sinn verlangen wir zusätzlich, dass **beide** Seiten Named Entities sind. Dann entsteht eine Kante zwischen zwei Akteuren: *die CSU kritisiert Merkel*, *Bayern besiegt Darmstadt*.

In [ ]:
# Liefere die Entity, zu der ein Token gehört (oder None)
def entity_of(token):
    for ent in token.doc.ents:
        if ent.start <= token.i < ent.end:
            return ent
    return None


# Sammle Relationen: Subjekt (sb) und Akkusativobjekt (oa) eines Verbs, beide Named Entities
ACTORS = {"PER", "ORG", "LOC"}
rows = []
for outlet, month, doc in zip(corpus["outlet"], corpus["month"], docs):
    for verb in doc:
        if verb.pos_ != "VERB":
            continue
        for subj in (c for c in verb.children if c.dep_ == "sb"):
            for obj in (c for c in verb.children if c.dep_ == "oa"):
                s, o = entity_of(subj), entity_of(obj)
                if s is not None and o is not None and s.label_ in ACTORS and o.label_ in ACTORS:
                    rows.append({"outlet": outlet, "month": month, "subject": s.text, "relation": verb.lemma_,
                                 "object": o.text, "text": doc.text})
relations = pd.DataFrame(rows)
print(f"{len(relations):,} Relationen zwischen zwei Entities")
relations["relation"].value_counts().head(15).to_frame("Anzahl").T

Die häufigsten Relationen stammen aus dem Sport: *schlagen*, *besiegen*, *bezwingen* verbinden zwei Mannschaften, und Sportmeldungen folgen einem festen Schema, das der Parser gut erkennt. Interessanter für die Politik ist *kritisieren*. Wer kritisiert wen?

In [ ]:
# Alle Relationen "kritisieren", mit dem Text, aus dem sie stammen
critics = relations[relations["relation"] == "kritisieren"]
critics[["outlet", "subject", "object", "text"]].sample(12, random_state=1)

Die Tabelle ist eine kleine Sammlung von **Konflikten**: Politiker kritisieren Politiker (Faymann - Merkel, Sanders - Clinton), Staaten andere Staaten (China - USA, USA - Russland), die EU-Kommission einen Mitgliedstaat. Lies die Spalte `text` mit: Ein Teil der Tripel ist falsch. *Stephan Weil kritisiert Wolfgang Schäuble* wurde als Schäuble → Weil extrahiert, weil der Parser bei Schlagzeilen ohne Artikel Subjekt und Objekt verwechselt (im Deutschen erkennt man sie oft nur am Kasus, und Eigennamen zeigen ihn nicht) oder weil die Entity-Erkennung nur einen Teil eines Namens erfasst. Wie in Skript 02 gilt: Das Ergebnis ist eine **Stichprobe überwiegend richtiger Fälle**, kein vollständiges und kein fehlerfreies Verzeichnis. Für eine Netzwerkanalyse - wer kritisiert wen am häufigsten? - bräuchte es mehr Text als Schlagzeilen und Teaser und eine Zusammenführung der Namensvarianten.

## Ereignisse: Was ist wo passiert?

Ein **Ereignis** besteht aus einem **Auslöser** (engl. *trigger*), der sagt, was passiert ist, und aus **Argumenten**: wo, wann, wer beteiligt war. Han et al. nennen als Beispiel *arrest a suspect*; in Nachrichten sind die Auslöser oft Substantive: *Anschlag*, *Erdbeben*, *Absturz*, *Streik*.

Wir definieren eine Liste von Auslösern. Weil das Deutsche Komposita bildet, zählt auch jedes Wort, das auf einen Auslöser *endet* - *Terroranschlag*, *Selbstmordanschlag* und *Flugzeugabsturz* sind dann automatisch dabei. Als Ort nehmen wir die erste Orts-Entity des Textes, als Zeit den Monat des Artikels.

In [ ]:
# Auslöser für Ereignisse (auch als letzter Teil eines Kompositums)
TRIGGERS = ["Anschlag", "Attentat", "Erdbeben", "Explosion", "Absturz", "Unwetter", "Streik",
            "Amoklauf", "Brand", "Schießerei", "Lawine", "Überschwemmung"]
trigger_re = re.compile(r"(?:" + "|".join(t.lower() for t in TRIGGERS) + r")$")

# Sammle für jeden Auslöser in einem Text: Auslöser, Ort (erste LOC-Entity), Monat, Zeitung
rows = []
for outlet, month, doc in zip(corpus["outlet"], corpus["month"], docs):
    locations = [e.text for e in doc.ents if e.label_ == "LOC"]
    for token in doc:
        if token.pos_ == "NOUN" and trigger_re.search(token.lemma_.lower()):
            rows.append({"outlet": outlet, "month": month, "trigger": token.lemma_,
                         "location": locations[0] if locations else None, "text": doc.text})
events = pd.DataFrame(rows)
print(f"{len(events):,} Ereignisnennungen, davon {events['location'].notna().mean():.0%} mit Ort")
events["trigger"].value_counts().head(15).to_frame("Anzahl").T

*Anschlag* und seine Komposita sind die mit Abstand häufigsten Auslöser - es war das Jahr der Anschläge des sogenannten Islamischen Staats in Europa. Verdichten wir die Anschläge zu einer Chronik: Wie oft wurde in welchem Monat über einen Anschlag an welchem Ort berichtet?

In [ ]:
# Nur Anschläge und Attentate mit Ort
attacks = events[events["trigger"].str.contains("nschlag|ttentat") & events["location"].notna()]
# Die zehn am häufigsten genannten Orte, Monat für Monat
top_places = attacks["location"].value_counts().head(10).index
chronicle = pd.crosstab(attacks["location"], attacks["month"].dt.strftime("%Y-%m")).loc[top_places]

# Zeichne die Chronik als Heatmap
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(chronicle, cmap="Reds", aspect="auto")
ax.set(xticks=range(chronicle.shape[1]), xticklabels=chronicle.columns, yticks=range(len(top_places)),
       yticklabels=top_places, title="Nennungen eines Anschlags nach Ort und Monat (beide Zeitungen)")
ax.tick_params(axis="x", rotation=45)
for (i, j), n in np.ndenumerate(chronicle.to_numpy()):
    if n:
        ax.text(j, i, n, ha="center", va="center", fontsize=7, color="white" if n > chronicle.to_numpy().max() / 2 else "black")
fig.tight_layout()
plt.show()

Ohne dass wir dem Verfahren je gesagt hätten, welche Anschläge es gab, rekonstruiert die Heatmap die Chronik des Jahres: Paris im November 2015, Brüssel im März 2016, die Anschlagsserie in der Türkei (Suruç im Juli, Ankara im Oktober 2015 sowie im Februar und März 2016) und Tunesien (Sousse im Juni). Auch Orte außerhalb Europas tauchen auf, Nigeria (Boko Haram) und Kabul, wenn auch mit wenigen Meldungen je Monat. Die Struktur kommt aus zwei einfachen Bausteinen - einer Wortliste und der Entity-Erkennung - und aus der Menge: Ein einzelner Fehler fällt nicht ins Gewicht, solange sich echte Ereignisse in vielen Texten wiederholen.

Auch hier lohnt der Blick auf die Fehler. Die Ortsangabe ist die *erste* Orts-Entity im Text, nicht unbedingt der Ort des Anschlags. *Frankreich* und *Paris* stehen für dasselbe Ereignis in getrennten Zeilen, und *Deutschland* sammelt Meldungen über Brandanschläge auf Flüchtlingsunterkünfte genauso wie Berichte über verhinderte Anschläge und Festnahmen. Und ein Text mit zwei Auslösern (*Terroranschlägen* und *Anschläge*) wird doppelt gezählt - wir zählen Nennungen, nicht Ereignisse. Die Ereignisextraktion in Forschung und Praxis löst das mit Modellen, die Auslöser und Argumente gemeinsam erkennen, und mit Wissensbasen, die *Paris* als Teil von *Frankreich* kennen - womit wir bei den Taxonomien sind.

In [ ]:
# Lies einige Anschlagsmeldungen, die "Deutschland" als Ort haben
attacks.loc[attacks["location"] == "Deutschland", ["outlet", "month", "text"]].head(6)

## Taxonomien aus Mustern

Eine **Taxonomie** ordnet Begriffe in Ober- und Unterbegriffe: *Griechenland* ist ein *Land*, *Google* ist ein *Unternehmen*. Taxonomien von Hand zu pflegen ist teuer, und jede Anwendung braucht eine etwas andere. Han et al. beschreiben die automatische Konstruktion von Taxonomien aus Text als eigene Aufgabe des Text Mining.

Das klassische Verfahren stammt von Marti Hearst (1992). Sie beobachtete, dass Texte Ober- und Unterbegriffe in wiederkehrenden Mustern nennen: *Länder wie Griechenland, Italien und Spanien*. Solche **Hearst-Muster** sind hochpräzise, aber selten. Schlagzeilen sind zu kurz dafür; wir brauchen die vollen Artikeltexte, die `tm.load_bodies()` liefert (für beide Zeitungen rund 20 Millionen Wörter).

Das Muster nutzt eine Eigenheit des Deutschen: Substantive und Namen beginnen mit einem Großbuchstaben. Der Oberbegriff ist ein großgeschriebenes Wort, dem ein kleingeschriebenes Wort oder ein Komma vorangeht (damit kein Satzanfang wie *So wie …* zum Oberbegriff wird), gefolgt von *wie* und einer Aufzählung großgeschriebener Wörter.

In [ ]:
# Lade die vollen Artikeltexte (doc_key -> Text)
bodies = tm.load_bodies()
print(f"{len(bodies):,} Artikel mit {bodies.str.split().str.len().sum():,} Wörtern")

# Ein großgeschriebenes Wort (Substantiv oder Name), optional aus zwei Wörtern ("Red Bull")
NOUN = r"[A-ZÄÖÜ][\wäöüß-]+"
ITEM = rf"{NOUN}(?: {NOUN})?"
# Oberbegriff + "wie" (+ optional "etwa"/"zum Beispiel") + Aufzählung "A, B und C"
HEARST = re.compile(
    rf"(?<=[a-zäöüß,] )({NOUN}) wie (?:etwa |zum Beispiel |beispielsweise )?((?:{ITEM}, )*{ITEM}(?: (?:und|oder) {ITEM})?)"
)

# Finde alle Treffer und zerlege die Aufzählung in einzelne Unterbegriffe
rows = []
for key, text in bodies.items():
    for m in HEARST.finditer(text):
        for hyponym in re.split(r", | und | oder ", m.group(2)):
            rows.append({"hypernym": m.group(1), "hyponym": hyponym, "match": m.group(0), "doc_key": key})
isa = pd.DataFrame(rows)
print(f"{len(isa):,} Paare aus Ober- und Unterbegriff")
isa.sample(8, random_state=0)[["hypernym", "hyponym", "match"]]

Zwei Probleme springen ins Auge. Der Oberbegriff steht in verschiedenen Kasus (*Länder*, *Ländern*), und manche Treffer sind keine Aufzählung von Beispielen, sondern ein Vergleich (*so groß wie Bayern*). Das erste lösen wir, indem wir eine Dativ-Endung *-n* abschneiden, wenn es die Form ohne *-n* auch gibt. Das zweite lässt sich mit einem regulären Ausdruck nicht lösen; dafür hilft die Häufigkeit: Ein Paar, das in mehreren Artikeln vorkommt, ist selten ein Zufallstreffer.

In [ ]:
# Führe Dativformen zusammen: "Ländern" -> "Länder", wenn "Länder" auch als Oberbegriff vorkommt
known = set(isa["hypernym"])
isa["hypernym"] = [h[:-1] if h.endswith("n") and h[:-1] in known else h for h in isa["hypernym"]]

# Zähle, in wie vielen Artikeln jedes Paar vorkommt
pairs = isa.groupby(["hypernym", "hyponym"])["doc_key"].nunique().rename("Artikel").reset_index()
print(f"{len(pairs):,} verschiedene Paare, davon {(pairs['Artikel'] >= 2).sum():,} in mindestens zwei Artikeln")
pairs.groupby("hypernym")["Artikel"].sum().sort_values(ascending=False).head(20).to_frame("Treffer").T

Unter den häufigsten Oberbegriffen stehen die Kategorien, in denen Journalisten denken: Länder, Staaten, Städte, Konzerne, Unternehmen, Spieler. Dazwischen stehen Wörter im Singular wie *Name*, *Marke* oder *Größe*. Sie stammen meist aus Vergleichen (*eine Marke wie Coca-Cola*, *eine Größe wie Deutschland*) und zeigen, dass das Muster nicht zwischen Beispiel und Vergleich unterscheidet.

Für einige Oberbegriffe sehen wir uns die Unterbegriffe an, die in mindestens zwei Artikeln vorkommen:

In [ ]:
# Unterbegriffe je Oberbegriff, nach Anzahl der Artikel
children = defaultdict(list)
for r in pairs.sort_values("Artikel", ascending=False).itertuples():
    children[r.hypernym].append((r.hyponym, r.Artikel))


# Gib die Unterbegriffe eines Oberbegriffs als Baum aus (nur Paare aus mindestens min_articles Artikeln)
def show_tree(term: str, width: int = 8, min_articles: int = 2) -> None:
    kids = [(k, n) for k, n in children.get(term, []) if n >= min_articles][:width]
    print(term)
    for i, (kid, n) in enumerate(kids):
        print(f"{'└──' if i == len(kids) - 1 else '├──'} {kid} ({n})")


for term in ["Schwellenländer", "Konzerne", "Netzwerke", "Krankheiten"]:
    show_tree(term)
    print()

Aus Mustern in freiem Text ist ein Stück Taxonomie entstanden, das ein Mensch auf Anhieb versteht: Schwellenländer sind China, Brasilien und Indien, Konzerne vor allem die großen Technologiefirmen, Netzwerke Facebook und Twitter.

Eine Taxonomie ist aber mehr als eine Liste von Paaren; sie ist eine **Hierarchie**. Was passiert, wenn wir die Paare zu einer Hierarchie verketten? Zwei Abfragen zeigen die Grenzen: die Oberbegriffe eines Begriffs, und die Ketten, in denen ein Unterbegriff selbst wieder Oberbegriff ist.

In [ ]:
# Unter welchen Oberbegriffen steht ein Begriff?
for term in ["Facebook", "China"]:
    parents = pairs[(pairs["hyponym"] == term) & (pairs["Artikel"] >= 2)].sort_values("Artikel", ascending=False)
    print(f"{term} ist ein/eine:", ", ".join(f"{h} ({n})" for h, n in zip(parents["hypernym"], parents["Artikel"])))

# Ketten: Ein Unterbegriff ist selbst Oberbegriff (beide Paare aus mindestens zwei Artikeln)
strong = pairs[pairs["Artikel"] >= 2]
strong[strong["hyponym"].isin(set(strong["hypernym"]))].sort_values("Artikel", ascending=False).head(6)

Beide Abfragen zeigen typische Eigenschaften automatisch gebauter Taxonomien:

- **Sie ist kein Baum.** *Facebook* ist ein Netzwerk, ein Konzern, eine Firma und eine Plattform, *China* ein Schwellenland, ein Land und ein Staat. Ein Begriff hat mehrere Oberbegriffe, die verschiedene Facetten beschreiben (Han et al. sprechen von *multifacettierten* Taxonomien). Welche Facette eine Anwendung braucht, entscheidet sie selbst.
- **Verkettung verstärkt Fehler.** Die Ketten sind fast alle unsinnig. *Frauen wie Männer* heißt "Frauen ebenso wie Männer" und ist keine Aufzählung von Beispielen, und weil auch *Männer wie Frauen* vorkommt, entsteht sogar ein Kreis. Eine echte Hierarchie über mehrere Ebenen braucht weitere Muster (*X und andere Y*, *Y, darunter X*), Regeln gegen Kreise oder Embeddings, die Begriffe auf derselben Ebene erkennen.
- **Sie spiegelt das Korpus.** Die Unterbegriffe sind die Länder, Firmen und Netzwerke, über die 2015/16 geschrieben wurde. Han et al. sehen darin den Vorteil: Jede Anwendung kann ihre eigene, zum Korpus passende Taxonomie bekommen.
- **Die Häufigkeit ist ein Qualitätssignal, aber kein Beweis.** Paare aus einem einzigen Artikel sind oft Fehltreffer, Paare aus vielen Artikeln meist richtig - aber auch häufige Paare können systematisch falsch sein, wie *Frauen wie Männer* zeigt. Übung 3 misst das.

## Argument Mining

Die bisherigen Verfahren extrahieren Fakten: wer, was, wo. **Argument Mining** extrahiert *Begründungen*. Ein Argument besteht im einfachsten Modell aus einer **Behauptung** (engl. *claim*) und einer oder mehreren **Prämissen**, die sie stützen. Die Aufgabe hat mehrere Stufen:

1. Enthält ein Text überhaupt Argumentation?
2. Welche Sätze oder Satzteile sind Behauptungen, welche Prämissen?
3. Welche Prämisse stützt (oder bestreitet) welche Behauptung?

Für die erste Stufe haben wir gelabelte Daten. Die Redaktion von Der Standard hat für das One Million Posts Corpus Leserkommentare annotiert, unter anderem mit der Kategorie **`ArgumentsUsed`**: Begründet der Kommentar seine Position?

### Die Daten: annotierte Leserkommentare

In [ ]:
# Lade die annotierten Leserkommentare und behalte die, die für ArgumentsUsed annotiert wurden
posts = tm.load_posts()
args = posts[posts["ArgumentsUsed"].notna()].reset_index(drop=True)
y = args["ArgumentsUsed"].astype(int).to_numpy()
print(f"{len(args):,} Kommentare, davon {y.mean():.0%} mit Argumenten")

# Zwei Beispiele je Klasse
for label in [1, 0]:
    print(f"\n--- ArgumentsUsed = {label}")
    for text in args.loc[y == label, "text"].sample(2, random_state=3):
        print(" -", text[:300])

Rund 28 % der Kommentare enthalten Argumente. Die Beispiele zeigen, wie die Redaktion annotiert hat: Ein argumentierender Kommentar begründet eine Position (*Man kann Frauen nicht aus jeder Verantwortung entlassen, selbst dann, wenn …*), ein nicht argumentierender stellt fest, fragt, spottet oder ruft etwas aus.

### Diskursmarker als Indikatoren

Die naheliegende Regel: Wer argumentiert, verwendet **Diskursmarker** wie *weil*, *denn*, *daher* oder *deshalb*. Wir prüfen für jeden Marker, wie oft ein Kommentar, der ihn enthält, argumentiert - im Vergleich zur Grundrate von 28 %.

In [ ]:
# Diskursmarker, die eine Begründung oder Folgerung einleiten, und einige für Einwände
MARKERS = ["weil", "denn", "da", "nämlich", "daher", "deshalb", "deswegen", "darum", "also", "somit",
           "obwohl", "trotzdem", "aber", "jedoch", "allerdings", "sondern", "wenn"]
lower = args["text"].str.lower()
pd.DataFrame(
    {m: {"Kommentare": lower.str.contains(rf"\b{m}\b").sum(),
         "davon mit Argumenten": y[lower.str.contains(rf"\b{m}\b").to_numpy()].mean()} for m in MARKERS}
).T.sort_values("davon mit Argumenten", ascending=False).style.format({"Kommentare": "{:.0f}", "davon mit Argumenten": "{:.0%}"})

Jeder Marker hebt den Anteil über die Grundrate, Folgerungsmarker wie *deswegen*, *daher* und *deshalb* auf mehr als das Doppelte. Die Regel scheint zu funktionieren. Aber bevor wir sie verwenden, sollten wir fragen, *was* sie eigentlich misst.

### Die Länge als Störfaktor

Ein langer Kommentar enthält mit größerer Wahrscheinlichkeit *irgendein* bestimmtes Wort - auch *weil*. Und ein langer Kommentar hat mehr Platz für eine Begründung. Wenn beides zutrifft, misst die Markerregel vielleicht nur die Länge.

In [ ]:
# Länge in Wörtern und ob einer der kausalen Marker vorkommt
CAUSAL = ["weil", "denn", "da", "nämlich", "daher", "deshalb", "deswegen", "darum", "also", "somit"]
args["words"] = args["text"].str.split().str.len()
args["marker"] = lower.str.contains(r"\b(?:" + "|".join(CAUSAL) + r")\b")

# Anteil argumentierender Kommentare nach Längenquartil, mit und ohne Marker
args["Länge"] = pd.qcut(args["words"], 4, labels=["sehr kurz", "kurz", "lang", "sehr lang"])
table = args.assign(arg=y).pivot_table(index="Länge", columns="marker", values="arg", aggfunc=["mean", "size"], observed=True)
table.columns = [f"{'Anteil' if a == 'mean' else 'Anzahl'} {'mit' if m else 'ohne'} Marker" for a, m in table.columns]
print("Länge in Wörtern je Quartil:", args.groupby("Länge", observed=True)["words"].agg(["min", "max"]).to_dict("index"))
table.style.format({c: "{:.0%}" for c in table.columns if c.startswith("Anteil")})

Das ist das wichtigste Ergebnis dieses Abschnitts. **Innerhalb jedes Längenquartils ändert der Marker fast nichts**: Sehr lange Kommentare argumentieren zu rund zwei Dritteln, ob sie *weil* enthalten oder nicht, sehr kurze fast nie. Marker kommen vor allem in langen Kommentaren vor, und lange Kommentare argumentieren. Der scheinbare Effekt der Marker war zum größten Teil ein Effekt der Länge - ein klassischer **Störfaktor** (engl. *confounder*).

Was heißt das für einen Klassifikator? Wir vergleichen vier Modelle mit zehnfacher Kreuzvalidierung. Die Kommentare verteilen sich über das ganze Jahr, mit Schwerpunkten im Juli 2015 und im April 2016. Anders als in Skript 05 geht es aber nicht um wechselnde Themen, sondern um die Art zu schreiben, die sich über die Zeit kaum ändert; eine geschichtete Zufallsaufteilung ist deshalb vertretbar. (Übung 6 prüft das mit einer zeitlichen Aufteilung.) Weil nur 28 % der Kommentare positiv sind, berichten wir neben dem F1-Wert für die Klasse *Argument* die **ROC-AUC**: die Wahrscheinlichkeit, dass das Modell einem zufälligen argumentierenden Kommentar einen höheren Wert gibt als einem zufälligen nicht argumentierenden.

In [ ]:
# Zehnfache, geschichtete Kreuzvalidierung
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)


def evaluate(model, X) -> dict:
    p = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]
    return {"F1 (Argument)": f1_score(y, p > 0.5), "ROC-AUC": roc_auc_score(y, p)}


# Merkmale: Länge (logarithmiert) und Marker
length = np.log1p(args[["words"]].to_numpy())
length_marker = np.c_[length, args["marker"].to_numpy()]
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2))
logreg = LogisticRegression(max_iter=3000, class_weight="balanced")

pd.DataFrame({
    "nur Marker (Regel)": {"F1 (Argument)": f1_score(y, args["marker"]), "ROC-AUC": roc_auc_score(y, args["marker"])},
    "nur Länge": evaluate(logreg, length),
    "Länge + Marker": evaluate(logreg, length_marker),
    "TF-IDF (Wörter und Bigramme)": evaluate(make_pipeline(tfidf, logreg), args["text"]),
}).T.round(3)

Die Länge allein erreicht eine ROC-AUC von über 0,8; der Marker fügt ihr praktisch nichts hinzu. Ein TF-IDF-Modell über alle Wörter ist nur wenig besser - und auch dieses Modell kennt die Länge: Ein langer Text hat mehr Merkmale ungleich null. Wer hier einen Klassifikator für Argumentation trainiert und eine AUC von 0,85 berichtet, sollte dazusagen, dass ein Lineal 0,82 schafft.

Das ist kein Fehler der Annotation. Begründen braucht Platz, und kurze Kommentare können meist nicht argumentieren. Die Lehre ist allgemeiner: **Prüfe, was ein Modell wirklich gelernt hat, mit einer einfachen Baseline aus naheliegenden Störgrößen** - Länge, Zeit, Quelle. In Skript 05 war es die Zeit, die eine zufällige Aufteilung verzerrt hat; hier ist es die Länge.

### Behauptung und Begründung in Kommentaren

Die zweite und dritte Stufe des Argument Mining - Behauptungen und Prämissen finden und verbinden - brauchen eigentlich Daten, die auf Satzebene annotiert sind. Ohne solche Daten bleibt ein bescheidener, regelbasierter Weg: Wo ein Diskursmarker steht, zerlegt er den Satz in zwei Teile mit bekannter Funktion.

- *X, **weil** Y*: X ist die Behauptung, Y die Prämisse. (*Denn* wäre ebenso naheliegend, ist aber mehrdeutig: *es sei denn*, *mehr denn je*.)
- ***Deshalb** X* (auch *daher*, *deswegen*, *darum*) am Satzanfang: X ist die Behauptung, der vorangehende Satz die Prämisse.

Als Texte nehmen wir die Kommentare der Redaktion von Der Standard - also die Meinungsartikel im Ressort *Kommentare*, nicht die Leserkommentare. Hier wird mit Absicht argumentiert, und der Text ist sorgfältig redigiert.

In [ ]:
# Die Meinungsartikel aus dem Ressort "Kommentare" mit ihrem vollen Text
opinion = corpus[(corpus["outlet"] == "Der Standard") & (corpus["path"].str.contains("/Kommentare/"))].copy()
opinion["body"] = opinion["doc_key"].map(bodies)
print(f"{len(opinion)} Kommentare der Redaktion")

# Ein einfacher Satzsegmentierer: ein leeres deutsches Modell mit regelbasierter Satzerkennung
segmenter = spacy.blank("de")
segmenter.add_pipe("sentencizer")

CONCLUSION = re.compile(r"^(Deshalb|Daher|Deswegen|Darum|Also),? ")
REASON = re.compile(r"^(?P<claim>.{15,}?), weil (?P<premise>.{15,})$")

# Suche in jedem Kommentar Paare aus Behauptung und Prämisse
rows = []
for r in opinion.itertuples():
    sentences = [s.text.strip() for s in segmenter(r.body).sents]
    for i, s in enumerate(sentences):
        if (m := REASON.match(s)):
            rows.append({"title": r.title, "Muster": "X, weil Y", "Behauptung": m["claim"], "Prämisse": m["premise"]})
        elif CONCLUSION.match(s) and i > 0:
            rows.append({"title": r.title, "Muster": "Y. Deshalb X", "Behauptung": s, "Prämisse": sentences[i - 1]})
arguments = pd.DataFrame(rows)
print(f"{len(arguments)} Paare aus {arguments['title'].nunique()} Kommentaren")
arguments["Muster"].value_counts()

In [ ]:
# Zeige einige gefundene Argumente
pd.set_option("display.max_colwidth", 160)
arguments.sample(8, random_state=2)[["Muster", "Behauptung", "Prämisse"]]

Viele dieser Paare sind echte kleine Argumente, wie sie ein Leitartikel aufbaut: eine Bewertung oder Forderung als Behauptung, ein Sachverhalt als Prämisse. Lies sie trotzdem kritisch:

- **Nicht jedes *weil* begründet eine Behauptung.** Oft erklärt es einen Sachverhalt (*Die Preise steigen, weil die Nachfrage wächst*) - eine Ursache, kein Argument. Der Unterschied liegt in der Funktion im Text, nicht im Wort.
- **Die meisten Argumente haben keinen Marker.** Ein Leitartikel argumentiert über ganze Absätze, und die Verbindung zwischen Prämisse und Behauptung bleibt meist implizit. Eine Markerregel findet nur die explizit markierte Spitze des Eisbergs - ihre *Precision* kann gut sein, ihr *Recall* ist gering.
- **Wessen Argument?** Ein Teil der Behauptungen steht im Konjunktiv (*die Kandidatur sei abzulehnen*): Der Kommentar gibt das Argument eines anderen wieder, oft um ihm zu widersprechen. Wer argumentiert, ist für die Analyse einer Debatte genauso wichtig wie was argumentiert wird.
- **Die Grenzen sind grob.** Eine Behauptung kann aus mehreren Sätzen bestehen, und die Prämisse vor *deshalb* steht nicht immer im unmittelbar vorangehenden Satz. Und nicht jedes *Also* am Satzanfang leitet eine Folgerung ein.

Forschungssysteme für Argument Mining trainieren deshalb Modelle auf Texten, in denen Menschen Behauptungen, Prämissen und ihre Beziehungen annotiert haben, heute meist durch Feintuning vortrainierter Sprachmodelle oder mit LLMs. Die Regeln hier zeigen, *was* diese Modelle lernen müssen - und warum die Länge eines Textes allein ein überraschend starker Konkurrent ist.

## Zusammenfassung

| Aufgabe | Verfahren hier | Baustein | typischer Fehler |
|---|---|---|---|
| Rollen (Person - Funktion) | Funktion als Kopf einer PER-Entity | Dependenzparsing, NER | Namensvarianten, gleiche Nachnamen |
| Relationen zwischen Entities | Subjekt und Objekt eines Verbs sind Entities | Dependenzparsing, NER | Subjekt und Objekt vertauscht |
| Ereignisse | Auslöserwörter + Ort + Monat | Wortliste, NER | falscher Ort, verhinderte Anschläge und Festnahmen, Doppelzählungen |
| Taxonomie | Hearst-Muster *Y wie X1, X2 und X3* | reguläre Ausdrücke | Vergleiche statt Beispiele, flache Hierarchie |
| argumentierende Texte erkennen | Marker, Länge, TF-IDF | Klassifikation | Länge als Störfaktor |
| Behauptung und Prämisse | Diskursmarker *weil*, *deshalb* | Satzsegmentierung, Regeln | Ursache statt Begründung; geringer Recall |

Informationsextraktion verwandelt Text in Tabellen. Jede Zeile einer solchen Tabelle sieht gleich vertrauenswürdig aus, egal wie sie entstanden ist. Die Gewohnheit aus Skript 02 gilt hier doppelt: **Lies die Texte, aus denen eine Zeile stammt**, bevor du die Tabelle auswertest.

## Übungen

1. **Rollen normalisieren.** Baue ein Wörterbuch, das Varianten derselben Rolle zusammenführt (*Bundeskanzlerin* → *Kanzlerin*, *US-Präsident* → *Präsident*). Wie viele verschiedene Rollen bleiben übrig? Hilft es, nur den letzten Teil des Kompositums zu nehmen (*Bundesfinanzminister* → *minister*)?
2. **Ein weiteres Ereignis.** Erstelle eine Chronik für *Streik* oder *Erdbeben*. Welche Streiks haben die Zeitungen 2015/16 beschäftigt, und berichten beide Zeitungen über dieselben?
3. **Precision der Taxonomie.** Ziehe 30 Paare aus Ober- und Unterbegriff, die nur in *einem* Artikel vorkommen, und 30, die in mindestens drei Artikeln vorkommen. Bewerte jedes Paar von Hand als richtig oder falsch. Wie groß ist der Unterschied?
4. **Weitere Hearst-Muster.** Ergänze die Muster *X und andere Y* und *Y, darunter X*. Wie viele neue Paare findest du, und wie viele davon sind schon bekannt?
5. **Argumente ohne Längeneffekt.** Trainiere den TF-IDF-Klassifikator nur auf Kommentaren mit 30 bis 80 Wörtern. Wie hoch ist die ROC-AUC jetzt? Welche Wörter haben die größten Koeffizienten?
6. **Eine zeitliche Aufteilung.** Trainiere den TF-IDF-Klassifikator auf den Kommentaren bis Ende 2015 (`args["created"]`) und teste ihn auf denen aus 2016. Wie stark sinkt die ROC-AUC gegenüber der Kreuzvalidierung?

## Quellen

* Han, J., Pei, J., Tong, H. (2023): *Data Mining: Concepts and Techniques*, 4. Aufl., Kap. 12.1. Morgan Kaufmann.
* Hearst, M. A. (1992): Automatic Acquisition of Hyponyms from Large Text Corpora. *Proceedings of COLING 1992*, 539-545.
* Lawrence, J., Reed, C. (2019): Argument Mining: A Survey. *Computational Linguistics* 45(4), 765-818.
* Schabus, D., Skowron, M., Trapp, M. (2017): One Million Posts: A Data Set of German Online Discussions. *Proceedings of SIGIR 2017*, 1241-1244.
* https://spacy.io/usage/linguistic-features
* https://spacy.io/api/sentencizer
* https://scikit-learn.org/stable/modules/cross_validation.html